* P10 = Identify the frequencies contained in a periodic signal.

  * A sensor records a signal composed of two periodic oscillations and random noise.
  * Use the following parameters:

```python
sampling_frequency = 100   # measurements per second
duration = 5               # seconds
frequency_1 = 2            # Hz
frequency_2 = 7            # Hz
```
- Generate the time values using `numpy.linspace()`.
- Generate the signal using:

  $$
  s(t)=2\sin(2\pi f_1t)+\sin(2\pi f_2t)+\text{noise}
  $$

- Generate the noise with a normal distribution using NumPy.
- Create a Pandas `DataFrame` containing:
    - Time.
    - Signal value.
- Calculate and print:
    - The mean value of the signal.
    - The standard deviation of the signal.
    - The maximum and minimum values.
- Use the Fast Fourier Transform to obtain the frequency spectrum.
    - Hint: use `numpy.fft.rfft()` and `numpy.fft.rfftfreq()`.
- Calculate the amplitude corresponding to each frequency.
- Ignore the frequency at 0 Hz.
- Identify the two frequencies with the largest amplitudes.
- Print the two detected dominant frequencies.
- Create two plots:
    - Signal as a function of time.
    - Amplitude as a function of frequency.
- Restrict the frequency plot to the interval from 0 to 15 Hz.
- Use `pathlib` to create a directory named `P10_results`.
- Save:
    - The signal data as `P10_signal.csv`.
    - The frequency spectrum as `P10_spectrum.csv`.
    - The signal plot as `P10_signal.png`.
    - The frequency-spectrum plot as `P10_spectrum.png`.
- In the final notebook cell, briefly explain what the peaks in the frequency spectrum represent.

In [2]:
# -----------------------------------------------------------------------------
# Import Libraries
# -----------------------------------------------------------------------------
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [6]:
# -----------------------------------------------------------------------------
# 1. Signal Generation
# -----------------------------------------------------------------------------
sampling_frequency = 100  # Measurements per second (Hz)
duration = 5  # Seconds
frequency_1 = 2  # Hz
frequency_2 = 7  # Hz

n_samples = sampling_frequency * duration
t = np.linspace(0, duration, n_samples, endpoint=False)

# Gaussian white noise generation
np.random.seed(42)  # Seed for reproducibility
noise = np.random.normal(loc=0.0, scale=1.0, size=n_samples)

# Signal equation: s(t) = 2*sin(2*pi*f1*t) + sin(2*pi*f2*t) + noise
signal = (
    2 * np.sin(2 * np.pi * frequency_1 * t)
    + np.sin(2 * np.pi * frequency_2 * t)
    + noise
)

In [31]:
# -----------------------------------------------------------------------------
# 2. DataFrame Creation & Descriptive Statistics
# -----------------------------------------------------------------------------
# Create Dataframe
df_signal = pd.DataFrame({"Time": t, 
                          "Signal": signal})

# Calculate descriptive statistics
mean = df_signal['Signal'].mean()
deviation = df_signal['Signal'].std()
max_value = df_signal['Signal'].max()
min_value = df_signal['Signal'].min()

# Print Signal Descriptive Statistics
print("Mean: ", mean)
print("Standard Deviation: ", deviation)
print("Maximum value:", max_value)
print("Minimum: ", min_value)


Mean:  0.0068379945886474685
Standard Deviation:  1.936891367515187
Maximum value: 4.933416968165355
Minimum:  -4.662948938289663


In [32]:
# -----------------------------------------------------------------------------
# 3. Fast Fourier Transform (FFT) & Dominant Frequency Peak Detection
# -----------------------------------------------------------------------------
# Real-valued Fast Fourier Transform
fft_values = np.fft.rfft(signal)
frequencies = np.fft.rfftfreq(n_samples, d=(1 / sampling_frequency))

# Amplitude normalization (multiply by 2/N to account for one-sided spectrum)
amplitudes = np.abs(fft_values) * (2 / n_samples)
df_spectrum = pd.DataFrame({"Frequency": frequencies, "Amplitude": amplitudes})

# Filter out DC component (0 Hz) and retrieve top 2 dominant frequencies
top_two = df_spectrum[df_spectrum["Frequency"] > 0].nlargest(2, "Amplitude")

# Print dominant frequencies without index, formatted to 4 decimal places
print(top_two.to_string(index=False, float_format="%.4f"))

 Frequency  Amplitude
    2.0000     2.1298
    7.0000     1.0326


In [37]:
# -----------------------------------------------------------------------------
# 4. Directory Management & CSV File Export
# -----------------------------------------------------------------------------
output_dir = Path("P10_results")
output_dir.mkdir(exist_ok=True)

# Export DataFrames to CSV format
df_signal.to_csv(output_dir / "P10_signal.csv")
df_spectrum.to_csv(output_dir / "P10_spectrum.csv")

# -----------------------------------------------------------------------------
# 5. Plot Generation & Export
# -----------------------------------------------------------------------------

# Plot 1: Time-Domain Signal
plt.figure(figsize=(6, 6))
plt.plot(
    df_signal["Time"],     # X-axis: Time in seconds
    df_signal["Signal"],   # Y-axis: Signal amplitude with added noise
    color="tab:blue",      # Standard Tableau blue color for high contrast
    lw=1                   # Line width set to 1 pt for clear noise rendering
)

plt.title("Time-Domain Signal s(t)")
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.grid(True, linestyle="--", alpha=0.6)
plt.savefig(output_dir / "P10_signal.png", bbox_inches="tight")
plt.close()

# Plot 2: Frequency Spectrum
plt.figure(figsize=(6, 6))
plt.stem(
    df_spectrum["Frequency"],  # X-axis: Discrete frequency values (Hz)
    df_spectrum["Amplitude"],  # Y-axis: Normalized peak amplitude
    linefmt="crimson",         # Crimson color for vertical stem lines from Y=0
    markerfmt="ro"             # Red circular markers at peak amplitudes
)
plt.xlim(0, 15)  
plt.title("Amplitude Spectrum")
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.grid(True, linestyle="--", alpha=0.6)
plt.savefig(output_dir / "P10_spectrum.png", bbox_inches="tight")
plt.close()